# Transform Constructors Data

1. Read bronze `constructors` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`)
1. Rename columns to make them more meaningful (`name` → `constructor_name`)
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `constructors` table


- ## Step 1 - Read Bronze Table Data

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
constructors_df = spark.table(bronze_table)

## Step 2 : Keep only columns required for Analytics(Drop URL)

In [0]:
constructors_selected_df = constructors_df.drop("url")

## Step 3 &4 - Standardize column name


In [0]:
constructors_renamed_df = (
    constructors_selected_df
    .withColumnsRenamed(
        {"constructorId":"constructor_id",
         "name":"constructor_name",
         }
    )
)

In [0]:

# display(constructors_renamed_df)

Databricks data profile. Run in Databricks to view.

## Step 5 - Filter out rows where primary key is NULL

In [0]:
constructors_valid_df = (
    constructors_renamed_df
    .filter(
        F.col("constructor_id").isNotNull()

    )
)


In [0]:
# display(constructors_valid_df)

## Step 6 - Remove Duplicates

In [0]:
constructors_distinct_df = constructors_valid_df.dropDuplicates(["constructor_id"])

In [0]:
# display(constructors_distinct_df)

## Step 7 - Transform required column values to titlecase

In [0]:
constructors_final_df = (
    constructors_distinct_df
    .withColumn("nationality", F.initcap(F.col("nationality")))
    
)

In [0]:
# display(constructors_final_df)

## Step 8 - Write data to silver table

In [0]:
(
    constructors_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
# display(spark.table(silver_table))